In [20]:
import sys, pandas
print(sys.executable)
print(pandas.__version__)

/home/onyxia/work/Ensae-projet-python-2026/.venv/bin/python
2.3.3


In [21]:
import requests
from bs4 import BeautifulSoup

URL = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"
reponse = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (projet etudiant ENSAE)"},
                       timeout=30)
print(reponse.status_code, len(reponse.text))


200 166359


In [22]:
soup = BeautifulSoup(reponse.text, "lxml")
for titre in soup.find_all("h4")[:4]:
    print(titre.get_text(strip=True), "|", titre.parent.get("class"))

2026 FOMC Meetings | ['panel-heading']
2025 FOMC Meetings | ['panel-heading']
2024 FOMC Meetings | ['panel-heading']
2023 FOMC Meetings | ['panel-heading']


In [23]:
mois = soup.find_all(class_=lambda c: c and "month" in c)
print(len(mois))
print(mois[0].parent.prettify()[:1500])

57
<div "="" class="row fomc-meeting">
 <div class="fomc-meeting__month col-xs-5 col-sm-3 col-md-2">
  <strong>
   January
  </strong>
 </div>
 <div class="fomc-meeting__date col-xs-4 col-sm-9 col-md-10 col-lg-1">
  27-28
 </div>
 <div class="col-xs-12 col-md-4 col-lg-2">
  <strong>
   Statement:
  </strong>
  <br/>
  <a href="/monetarypolicy/files/monetary20260128a1.pdf">
   PDF
  </a>
  |
  <a href="/newsevents/pressreleases/monetary20260128a.htm">
   HTML
  </a>
  <br/>
  <a href="/newsevents/pressreleases/monetary20260128a1.htm">
   Implementation Note
  </a>
 </div>
 <div class="col-xs-12 col-md-4 col-lg-3">
  <a href="/monetarypolicy/fomcpressconf20260128.htm">
   Press Conference
  </a>
  <br/>
  <br/>
  <a href="/newsevents/pressreleases/monetary20260128b.htm">
   Statement on Longer-Run Goals and Monetary Policy Strategy
  </a>
 </div>
 <div class="col-xs-12 col-md-4 col-lg-4 fomc-meeting__minutes">
  <strong>
   Minutes:
  </strong>
  <br/>
  <a href="/monetarypolicy/files/fo

In [24]:
import re
import pandas as pd

MOIS = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}


def date_decision(annee, mois_txt, jours_txt):
    """Renvoie le dernier jour de la réunion (jour de l'annonce)."""
    mois_fin = mois_txt.split("/")[-1].strip()[:3].lower()
    jour_fin = int(re.findall(r"\d+", jours_txt)[-1])
    return pd.Timestamp(annee, MOIS[mois_fin], jour_fin)


def dates_fomc_recentes(html, debut=2021, fin=2025):
    """Extrait les dates de décision de la page actuelle du calendrier du FOMC."""
    soup = BeautifulSoup(html, "lxml")
    lignes = []
    for titre in soup.find_all("h4"):
        trouve = re.search(r"(\d{4}) FOMC Meetings", titre.get_text())
        if not trouve or not debut <= int(trouve.group(1)) <= fin:
            continue
        annee = int(trouve.group(1))
        panneau = titre.find_parent(class_="panel")
        for mois in panneau.find_all(class_="fomc-meeting__month"):
            jours = mois.find_next_sibling(class_="fomc-meeting__date")
            texte = jours.get_text(" ", strip=True)
            if "notation vote" in texte.lower():
                continue
            date = date_decision(annee, mois.get_text(strip=True), texte)
            heure = (date + pd.Timedelta(hours=14)).tz_localize("America/New_York")
            lignes.append({"date": date.date(), "banque": "Fed",
                           "heure_paris": heure.tz_convert("Europe/Paris").strftime("%H:%M"),
                           "type": "prevue"})
    return pd.DataFrame(lignes).sort_values("date").reset_index(drop=True)

In [25]:
fed = dates_fomc_recentes(reponse.text)
print(len(fed))
fed.head(10)

40


/tmp/ipykernel_60778/2666806210.py:31: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  heure = (date + pd.Timedelta(hours=14)).tz_localize("America/New_York")


,date,banque,heure_paris,type
0,2021-01-27,Fed,20:00,prevue
1,2021-03-17,Fed,19:00,prevue
2,2021-04-28,Fed,20:00,prevue
3,2021-06-16,Fed,20:00,prevue
4,2021-07-28,Fed,20:00,prevue
5,2021-09-22,Fed,20:00,prevue
6,2021-11-03,Fed,19:00,prevue
7,2021-12-15,Fed,20:00,prevue
8,2022-01-26,Fed,20:00,prevue
9,2022-03-16,Fed,19:00,prevue
